# Notebook 09D — Scoring Population and Model Selection Decision

**Goal:** Resolve the outstanding 09C questions without overwriting the Notebook 09 model or its 2024 investigation queue.

This notebook (1) profiles filing/source patterns associated with missing labels, (2) creates separate known/unknown scoring cohorts, (3) tests convergence-safe logistic alternatives using **2010–2020 training / 2021–2022 validation only**, and (4) produces a documented **conditional** decision. No 2023–2024 outcomes are used to choose a model. A positive label is a **self-reported significant diversion of assets**, not verified fraud. Model scores are not calibrated probabilities.

Run all cells from `notebooks/` or the project root in the `irs990` environment. Default model-fitting is enabled and may take several minutes. This notebook writes only to `data/processed/notebook09d/`.

## 1. Setup and input checks

In [ ]:
from pathlib import Path
import json, gc, time, warnings
import numpy as np
import pandas as pd
import duckdb
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
INPUT = ROOT / 'data/processed/nccs_longitudinal_features_v1.parquet'
RAW_DIR = ROOT / 'data/external/nccs/efile_v3_1'
N9 = ROOT / 'data/processed/notebook09'
N9C = ROOT / 'data/processed/notebook09c'
OUT = ROOT / 'data/processed/notebook09d'
OUT.mkdir(parents=True, exist_ok=True)
RUN_MODEL_FITS = True
TRAIN_CAP = 1_000_000
SEED = 42
for p in (INPUT, RAW_DIR, N9 / 'model_manifest.json'):
    assert p.exists(), f'Missing required input: {p}'
manifest = json.loads((N9 / 'model_manifest.json').read_text())
FEATURES = list(manifest['features'])
assert manifest['train_years'] == [2010, 2020]
assert manifest['validation_years'] == [2021, 2022]
assert 'fraud_label' not in FEATURES

def sqlpath(p):
    return str(p).replace("'", "''")

con = duckdb.connect()
con.execute('SET threads = 4')
con.execute("SET memory_limit = '6GB'")
con.execute(f"CREATE VIEW features AS SELECT * FROM read_parquet('{sqlpath(INPUT)}')")
columns = set(con.execute('DESCRIBE features').df().column_name)
assert set(FEATURES + ['OBJECTID','ein','tax_year','fraud_label']).issubset(columns)
print('Project:', ROOT)
print('Features:', len(FEATURES), 'Output:', OUT)
print('Model fits enabled:', RUN_MODEL_FITS)

## 2. Identify the source-data pattern without assuming a filing type

The 09C audit established that unknown labels and absent assets/governance are linked. This section examines the raw 2024 HEADER schema and creates an **observed** cohort summary. Header column names differ across NCCS extracts, so any filing-type field is discovered and displayed rather than assumed.

In [ ]:
header_files = sorted(p for p in RAW_DIR.rglob('*.parquet') if 'HEADER' in p.name.upper() and '2024' in p.name)
assert header_files, 'No 2024 HEADER Parquet found'
paths = '[' + ','.join("'" + sqlpath(p) + "'" for p in header_files) + ']'
con.execute(f"CREATE VIEW header24 AS SELECT * FROM read_parquet({paths}, union_by_name=true)")
hcols = con.execute('DESCRIBE header24').df().column_name.tolist()
print('HEADER columns:', hcols)
candidates = [c for c in hcols if any(t in c.lower() for t in ('form','return','type','subtype','version','schedule')) and c != 'OBJECTID']
print('Possible filing-type/schema fields:', candidates)

cohort = con.execute('''
    SELECT CASE WHEN fraud_label IN (0,1) THEN 'known' ELSE 'unknown' END AS label_status,
           COUNT(*) AS filings,
           COUNT(*) FILTER (WHERE total_assets IS NULL) AS assets_null,
           COUNT(*) FILTER (WHERE voting_members IS NULL) AS voting_null,
           COUNT(*) FILTER (WHERE total_revenue IS NULL) AS revenue_null
    FROM features WHERE tax_year=2024 GROUP BY 1 ORDER BY 1
''').df()
display(cohort)
cohort.to_csv(OUT/'01_2024_cohort_summary.csv',index=False)

# Only summarize source fields that actually exist.
for c in candidates[:8]:
    qc = '"' + c.replace('"','""') + '"'
    try:
        q = f"""
        SELECT CASE WHEN f.fraud_label IN (0,1) THEN 'known' ELSE 'unknown' END label_status,
               CAST(h.{qc} AS VARCHAR) AS header_value, COUNT(*) filings
        FROM features f LEFT JOIN header24 h ON f.OBJECTID=h.OBJECTID
        WHERE f.tax_year=2024 GROUP BY 1,2 ORDER BY 1,3 DESC
        """
        d = con.execute(q).df()
        d.to_csv(OUT/f'02_header_{c[:45].replace("/","_")}.csv',index=False)
        print('Header field:',c)
        display(d.groupby('label_status',group_keys=False).head(10))
    except Exception as e:
        print('Skipped header field',c,':',str(e)[:200])

**Interpretation:** A HEADER field may describe return form, extract format, or other metadata. Treat associations as diagnostic, not proof of the missingness cause. If no useful field is available, the underlying source mechanism remains unresolved.

## 3. Separate eligible and unvalidated 2024 populations

In [ ]:
# Store identifiers and eligibility status, not a new model score.
con.execute(f"""
    COPY (
      SELECT OBJECTID, ein, tax_year,
             CASE WHEN fraud_label IN (0,1) THEN 'known_label' ELSE 'unknown_label' END AS cohort,
             CASE WHEN fraud_label IN (0,1) AND total_assets IS NOT NULL
                       AND voting_members IS NOT NULL THEN TRUE ELSE FALSE END AS eligible_for_labeled_validation,
             total_assets IS NULL AS missing_assets,
             voting_members IS NULL AS missing_governance
      FROM features WHERE tax_year=2024
    ) TO '{sqlpath(OUT/'03_2024_scoring_cohorts.parquet')}' (FORMAT PARQUET)
""")
cohort_check=con.execute(f"""
    SELECT cohort, eligible_for_labeled_validation, COUNT(*) filings
    FROM read_parquet('{sqlpath(OUT/'03_2024_scoring_cohorts.parquet')}')
    GROUP BY 1,2 ORDER BY 1,2
""").df()
display(cohort_check)
assert cohort_check.filings.sum()==con.execute('SELECT COUNT(*) FROM features WHERE tax_year=2024').fetchone()[0]

## 4. Model candidates: converged logistic configurations

We compare **standardized**, imputed logistic regression with regularization strengths `C = 0.01, 0.1, 1.0` using the same sampled development population. These candidates are not guaranteed to beat 09B; the purpose is to find stable, reproducible alternatives. The previous 09C `liblinear` model is retained as a *historical reference*, not silently treated as converged. The training sample retains every positive and samples negatives with seed 42.

**Important:** Training uses case-control sampling and class balancing; scores must not be interpreted as real-world probabilities.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.exceptions import ConvergenceWarning

select_cols = ['tax_year','ein','OBJECTID','fraud_label'] + FEATURES
quoted = ','.join('"'+x.replace('"','""')+'"' for x in select_cols)
def load_dev(lo,hi):
    df=con.execute(f"""SELECT {quoted} FROM features
        WHERE tax_year BETWEEN ? AND ? AND fraud_label IN (0,1)
        ORDER BY tax_year,ein,OBJECTID""",[lo,hi]).df()
    for f in FEATURES:
        df[f]=pd.to_numeric(df[f],errors='coerce').replace([np.inf,-np.inf],np.nan).astype('float32')
    assert not df.duplicated(['ein','tax_year']).any()
    return df

def evaluate(y, scores):
    y=np.asarray(y,dtype=np.int8)
    scores=np.asarray(scores,dtype=float)
    assert np.isfinite(scores).all()
    order=np.argsort(-scores,kind='stable')
    result={'n':len(y),'positives':int(y.sum()),'prevalence':float(y.mean()),
            'average_precision':float(average_precision_score(y,scores)),
            'roc_auc':float(roc_auc_score(y,scores)) if len(np.unique(y))==2 else np.nan}
    for k in (100,250,500,1000):
        result[f'found_at_{k}']=int(y[order[:k]].sum())
        result[f'precision_at_{k}']=result[f'found_at_{k}']/min(k,len(y))
    return result

if RUN_MODEL_FITS:
    train=load_dev(2010,2020)
    val=load_dev(2021,2022)
    positives=train.loc[train.fraud_label.eq(1)]
    negatives=train.loc[train.fraud_label.eq(0)]
    assert len(positives)<=TRAIN_CAP
    nneg=TRAIN_CAP-len(positives)
    negatives=negatives.sample(n=min(nneg,len(negatives)),random_state=SEED)
    fit=pd.concat([positives,negatives],ignore_index=True).sample(frac=1,random_state=SEED)
    yval=val.fraud_label.astype('int8').to_numpy()
    print('Fit:',len(fit),'positive:',len(positives),'Validation:',len(val),'positive:',int(yval.sum()))

## 5. Fit, monitor convergence, and compare by validation year

In [ ]:
rows=[]
year_rows=[]
if RUN_MODEL_FITS:
    for C in (0.01,0.1,1.0):
        name=f'lbfgs_standard_C{C}'
        pipe=Pipeline([
            ('impute',SimpleImputer(strategy='median',add_indicator=True,keep_empty_features=True)),
            ('scale',StandardScaler()),
            ('model',LogisticRegression(C=C,solver='lbfgs',class_weight='balanced',
                                        max_iter=1500,random_state=SEED))
        ])
        start=time.monotonic()
        with warnings.catch_warnings(record=True) as w:
            warnings.simplefilter('always',ConvergenceWarning)
            pipe.fit(fit[FEATURES],fit.fraud_label.astype('int8'))
        convergence=[str(x.message) for x in w if issubclass(x.category,ConvergenceWarning)]
        niter=int(np.max(pipe.named_steps['model'].n_iter_))
        score=pipe.predict_proba(val[FEATURES])[:,1]
        row={'candidate':name,'C':C,'solver':'lbfgs','n_iter':niter,
             'convergence_warnings':len(convergence),'warning_text':' | '.join(convergence),
             'elapsed_sec':round(time.monotonic()-start,1),**evaluate(yval,score)}
        rows.append(row)
        for year in (2021,2022):
            mask=val.tax_year.eq(year).to_numpy()
            year_rows.append({'candidate':name,'year':year,**evaluate(yval[mask],score[mask])})
        pd.DataFrame(rows).to_csv(OUT/'04_candidate_validation.csv',index=False)
        pd.DataFrame(year_rows).to_csv(OUT/'05_candidate_by_year.csv',index=False)
        print(name,'AP',round(row['average_precision'],6),'top500',row['found_at_500'],
              'warnings',len(convergence),'n_iter',niter,flush=True)
        del pipe,score
        gc.collect()
    candidate_results=pd.DataFrame(rows).sort_values('average_precision',ascending=False)
    display(candidate_results[['candidate','average_precision','roc_auc','found_at_100','found_at_500',
                               'found_at_1000','n_iter','convergence_warnings']])
    display(pd.DataFrame(year_rows)[['candidate','year','average_precision','found_at_500','found_at_1000']])

## 6. Compare against 09C and document the decision

The 09C results are read from disk if present. **Do not choose a nonconverged model solely because its AP is slightly higher.** The decision is conditional: even a converged model with better validation ranking is **not validated for the unknown-label cohort**. No new 2024 ranking is generated here.

In [ ]:
prior_path=N9C/'06_logistic_stabilization_validation.csv'
prior=pd.read_csv(prior_path) if prior_path.exists() else pd.DataFrame()
if not prior.empty:
    display(prior[[c for c in ['candidate','average_precision','roc_auc','found_at_500','found_at_1000',
                               'convergence_warnings'] if c in prior.columns]])
else:
    print('09C comparison CSV not found; comparison is limited to current candidates.')

if RUN_MODEL_FITS:
    stable=candidate_results.loc[candidate_results.convergence_warnings.eq(0)].copy()
    if stable.empty:
        recommendation='No warning-free logistic candidate. Retain Notebook 09 baseline; investigate optimization.'
        selected=None
    else:
        selected=stable.sort_values('average_precision',ascending=False).iloc[0]
        recommendation=('Provisional warning-free logistic candidate identified for labeled population. '
                        'Compare ranking tradeoffs and year stability; do not deploy automatically.')
    decision={
        'training_years':[2010,2020], 'validation_years':[2021,2022],
        'test_period_not_used_for_selection':[2023,2024],
        'chosen_candidate':None if selected is None else str(selected['candidate']),
        'chosen_validation_average_precision':None if selected is None else float(selected['average_precision']),
        'chosen_found_at_500':None if selected is None else int(selected['found_at_500']),
        'known_label_population':'Eligible for retrospective labeled validation; not verified fraud.',
        'unknown_label_population':'Unvalidated: segregate and do not mix into a common prioritized queue.',
        'source_mechanism':'Source NULLs verified in 09C; exact cause remains unproven unless supported by HEADER results.',
        'model_scores_calibrated':False,
        'replace_notebook09_model':False,
        'replace_existing_2024_queue':False,
        'recommendation':recommendation,
    }
    (OUT/'06_scoring_and_model_decision.json').write_text(json.dumps(decision,indent=2))
    display(pd.DataFrame([decision]).T.rename(columns={0:'decision'}))

## Interpretation and Next Steps

### 1. Root Cause of Missingness: Form 990 vs. Form 990-EZ

Notebook 09D identified the primary explanation for the systematic missingness observed in Notebooks 09B and 09C.

The 2024 dataset contains two distinct filing populations:

| Filing Type | 2024 Filings | Disclosure Label | Financial and Governance Coverage |
|---|---:|---|---|
| Form 990 | 355,973 | Known (0 or 1) | Assets and governance fields populated |
| Form 990-EZ | 205,447 | Unknown (NULL) | Assets and governance fields missing in the current feature matrix |
| **Total** | **561,420** | | |

The separation is complete in the 2024 dataset: all Form 990 filings have known diversion-disclosure labels, while all Form 990-EZ filings have unknown labels.

This explains why approximately 36.6% of 2024 filings lack the financial and governance fields used by the current model.

**Key finding:** The missingness is structural and associated with filing type, rather than being caused by errors in the feature-engineering pipeline.

Form 990-EZ has different reporting requirements and does not provide the same set of fields as the full Form 990.

Notebook 09A independently confirmed that populated source values were preserved correctly and that missing values were not incorrectly converted into zeros in the six audited financial and governance fields.

The exact return-type distribution across all historical years has not yet been independently verified.

### 2. Model Stabilization Results

Notebook 09D tested three Logistic Regression configurations using the `lbfgs` solver and different regularization strengths.

All three configurations converged successfully.

| Model | Validation Average Precision | ROC-AUC | Positives @ 500 | Positives @ 1,000 | Converged |
|---|---:|---:|---:|---:|---|
| Logistic Regression, C=0.01 | 0.002830 | 0.6191 | 10 | 12 | Yes |
| Logistic Regression, C=0.1 | 0.002853 | 0.6193 | 10 | 12 | Yes |
| **Logistic Regression, C=1.0** | **0.002853** | **0.6194** | **10** | **12** | **Yes** |
| Previous Logistic Regression, `liblinear` | 0.004055 | 0.6031 | 11 | 11 | No |
| Notebook 09 Gradient Boosting | 0.002251 | 0.6642 | 7 | 11 | Yes |

The `lbfgs` model with C=1.0 was selected as the strongest converged Logistic Regression candidate.

However, the earlier `liblinear` configuration achieved higher validation Average Precision and identified more positive disclosures in the top 500, despite failing to converge.

Gradient Boosting retained the highest ROC-AUC, indicating stronger overall discrimination across the validation population.

**Conclusion:** The converged Logistic Regression model is a reasonable candidate for further evaluation, but no model has demonstrated consistent superiority across all metrics.

### 3. Model Selection Decision

The project will retain two primary modeling candidates:

1. **Histogram Gradient Boosting:** The original Notebook 09 model, which achieved stronger overall ROC-AUC.
2. **Converged Logistic Regression (`lbfgs`, C=1.0):** The stabilized Notebook 09D candidate, which achieved better validation Average Precision and top-K performance.

The unconverged `liblinear` model will remain documented as an experimental result but will not be adopted as the production candidate.

Model comparisons will continue to prioritize Average Precision and positive disclosures captured in the highest-ranked filings, while considering ROC-AUC, convergence, and year-by-year stability.

The 2023–2024 holdout has already been examined during development and cannot be treated as an untouched independent test of subsequent modeling decisions.

### 4. Scoring Population Decision

The project will separate filings into two populations.

**Primary population: Form 990**

- Use full Form 990 filings for supervised diversion-disclosure modeling.
- Continue developing risk rankings, peer comparisons, and model explanations.
- Evaluate model performance against known self-reported significant diversion disclosures.
- Retain both primary modeling candidates until further validation supports a final selection.

**Separate population: Form 990-EZ**

- Exclude Form 990-EZ filings from the primary validated-model investigation queue.
- Preserve the records for future analysis.
- Do not interpret existing model scores for these filings as validated or directly comparable with Form 990 scores.
- Consider a separate Form 990-EZ modeling approach only if an appropriate outcome label and compatible feature set become available.

The existing Notebook 09 investigation queue remains a retrospective research artifact and should not be used as a validated real-world fraud investigation tool.

### 5. Limitations

Several limitations remain:

- **Label definition:** The target represents self-reported significant diversion of assets, not independently confirmed fraud.
- **Rare outcomes:** Positive disclosures occur in fewer than 0.1% of labeled filings, making ranking metrics sensitive to small changes.
- **Generalizability:** Model performance on Form 990 filings cannot be assumed to apply to Form 990-EZ filings.
- **Point-in-time availability:** The current analysis uses retrospectively selected filings and historical features. A filing-time application would require additional availability-date checks.
- **Model calibration:** Risk scores are not validated probabilities of fraud.
- **Independent evaluation:** Further assessment requires a genuinely independent later period or other prespecified external validation.

### 6. Next Steps: Notebook 10

**Notebook 10 — Peer Benchmarking, Model Explanations, and Investigation Queue**

Notebook 10 will focus exclusively on Form 990 filings and will:

1. Develop peer groups using organizational characteristics such as size, revenue, and financial structure.
2. Calculate peer-relative financial and governance indicators.
3. Add model explanations to identify which features contribute to an organization's risk ranking.
4. Compare the original Gradient Boosting and converged Logistic Regression candidates.
5. Generate an analyst-friendly investigation queue containing organization identifiers, risk scores, peer comparisons, and explanatory indicators.
6. Clearly distinguish model-generated risk indicators from documented evidence of misconduct.

### Final Conclusion

Notebook 09D resolved the primary 2024 missingness question by identifying the structural distinction between Form 990 and Form 990-EZ filings.

It also established a converged Logistic Regression candidate without demonstrating that any single model consistently dominates across all evaluation metrics.

**Decision:** Preserve Notebooks 09–09D and their existing outputs, restrict the primary modeling population to Form 990 filings, retain both principal model candidates, and proceed to Notebook 10.

No additional Notebook 09 experiments are required before beginning the next phase.